# Section 7 — MCP, A Quick Demo (~15 min)

## Storyline

Until now PolicyPal's tools have been Python functions decorated with `@tool` — same process, same code. That's fine for a workshop, but in real Athora Netherlands the policy data lives behind internal APIs owned by other teams. You don't want to copy those APIs into every agent.

**Model Context Protocol (MCP)** is the answer: a small, standard protocol where a *server* exposes a set of tools, and any *client* (agent runtime, IDE, …) can call them. Agent Framework speaks MCP natively.

## What you'll see

| Concept | One-liner |
|---------|-----------|
| `MCPStreamableHTTPTool` | Connects an agent to an HTTP MCP server; tools are discovered automatically. |
| `header_provider` | Per-call auth headers (API key, bearer, …) without leaking secrets into shared state. |
| `function_invocation_kwargs` | How the agent caller passes per-call context (e.g., the API key) to the tool. |

**Time-box:** 15 minutes. Just enough to show the shape and answer *"can we plug PolicyPal into an internal API later?"* — yes, via an MCP server. Reference: [`microsoft/agent-framework — python/samples/mcp`](https://github.com/microsoft/agent-framework/tree/main/python/samples/mcp).

## MCP tool wiring (skeleton)

We won't actually run a live MCP server in the workshop. The code below is the shape Athora Netherlands would use against an internal authenticated MCP endpoint. The pattern follows `python/samples/02-agents/mcp/mcp_api_key_auth.py`.

In [ ]:
import os
from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()

MCP_URL = os.environ.get("ATHORA_MCP_URL", "https://policy-mcp.athora-netherlands.example/mcp")

credential = AzureCliCredential()
client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=credential,
)

policy_mcp_tool = MCPStreamableHTTPTool(
    name="athora_policy_api",
    description="Athora Netherlands internal policy API exposed over MCP.",
    url=MCP_URL,
    header_provider=lambda kwargs: {"Authorization": f"Bearer {kwargs['athora_api_key']}"},
)

policypal_mcp = Agent(
    client=client,
    name="PolicyPal-MCP",
    instructions=(
        "You are PolicyPal, an internal assistant for Athora Netherlands reps. "
        "Your policy lookup tools come from the connected MCP server. Use them; do not invent data."
    ),
    tools=[policy_mcp_tool],
)

# With a real MCP endpoint and token:
# result = await policypal_mcp.run(
#     "Look up policy NL-2031-887 for the rep.",
#     function_invocation_kwargs={"athora_api_key": "<dev-token>"},
# )
# print(result.text)
print("MCP-wired agent defined. Provide a real MCP_URL + token to run.")

### Walk-through: what happens at run time

1. The agent connects to the MCP server (HTTP streaming) and **discovers** its available tools.
2. The model now sees those tools in its tool list — same shape as if you'd defined them with `@tool` locally.
3. When the model decides to call one, the runtime sends an HTTP request to the MCP server, with headers built by `header_provider(kwargs)`.
4. The server executes the tool and returns the result; the model gets the result and continues.

Athora Netherlands' reality:
- One MCP server per data domain (policies, claims, billing) owned by the team that owns the data.
- Auth is enforced at the MCP server, not in the agent code.
- Multiple agents (PolicyPal, ClaimsAgent, …) reuse the same MCP servers — no copy-paste of API clients.

## When NOT to use MCP

- The tool is genuinely local logic (formatting, calculation) — keep it as a Python `@tool`.
- The tool needs callbacks into the agent's local state — MCP is request/response.
- You don't have an MCP server yet. Build it only when more than one agent (or one IDE/inspector) needs the same tool.

## Recap and what's next

- MCP = a tool-server protocol. Agent Framework supports it via `MCPStreamableHTTPTool` (and `MCPStdioTool` for local processes).
- Auth via `header_provider` keeps secrets out of shared state.
- For Athora Netherlands, MCP is the right pattern when an internal API needs to be reused by multiple agents.

**Coming up in Section 8 (Wrap-up):** hosting options for everything you've built today — Azure Functions, Container Apps, Durable Tasks — and what would change if/when Athora Netherlands gets approval to use hosted agents.